# Baseline Modeling - Final Version

## Konfiguration:
- **Daily Patterns**: Test Period = 4 Wochen, Gaps gefüllt mit 0
- **Weekly Patterns**: Test Period = 52 Wochen, Gaps als NaN (SARIMA interpoliert)

## Features:
- ✅ Gap Detection & unterschiedliches Filling
- ✅ Plotly Interactive Plots
- ✅ SARIMA vs. Seasonal Naive
- ✅ MAE & R² Metrics
- ✅ Summary Dashboard

In [1]:
# IMPORTS
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, r2_score
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive

from Favorita_TSA.models.data_preparation import build_dataframes
from Favorita_TSA.viz.ploty_theme import set_plotly_theme

set_plotly_theme()
print("✅ Imports loaded")

/Users/patrickhederer/Python Projects/Group-Work-Favorita-Forecasting/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports loaded


In [2]:
# KONFIGURATION

ITEMS_TO_MODEL = {
    "daily_smooth": {"store": 25, "item": 115611},
    "daily_erratic": {"store": 44, "item": 103520},
    "weekly_smooth": {"store": 24, "item": 1503844},
    "weekly_erratic": {"store": 51, "item": 1239986},
}

# WICHTIG: Unterschiedliche Test Periods!
TEST_WEEKS_DAILY = 4  # Daily: 4 Wochen = 28 Tage
TEST_WEEKS_WEEKLY = 52  # Weekly: 52 Wochen = 1 Jahr

# Plotly Theme
TEMPLATE = "plotly_white"  # oder "plotly_dark", "seaborn"

print("✅ Konfiguration geladen")
print(f"   Test Period daily:  {TEST_WEEKS_DAILY} Wochen")
print(f"   Test Period weekly: {TEST_WEEKS_WEEKLY} Wochen")
print(f"   Plotly Template: {TEMPLATE}")

✅ Konfiguration geladen
   Test Period daily:  4 Wochen
   Test Period weekly: 52 Wochen
   Plotly Template: plotly_white


In [3]:
# HELPER FUNCTIONS


def detect_gaps(df, date_col="ds", freq="D"):
    """Findet zeitliche Lücken."""
    df = df.sort_values(date_col)

    expected_dates = pd.date_range(
        start=df[date_col].min(), end=df[date_col].max(), freq=freq
    )

    actual_dates = set(df[date_col])
    missing_dates = sorted(set(expected_dates) - actual_dates)

    gap_info = {
        "has_gaps": len(missing_dates) > 0,
        "n_missing": len(missing_dates),
        "missing_dates": missing_dates[:10],
        "pct_missing": len(missing_dates) / len(expected_dates) * 100,
    }

    if gap_info["has_gaps"]:
        print(
            f"  ⚠️  Gaps: {gap_info['n_missing']} dates ({gap_info['pct_missing']:.1f}%)"
        )
    else:
        print("  ✅ No gaps")

    return gap_info


def fill_gaps(df, date_col="ds", target_col="y", freq="D"):
    """
    Füllt zeitliche Lücken.

    WICHTIG - Unterschiedliche Strategie:
      - Daily (freq='D'):  Füllt mit 0 (Store geschlossen/Feiertag)
      - Weekly (freq='W'): Lässt NaN (SARIMA interpoliert)

    Begründung:
      - 1 fehlender Tag = realistisch (Sonntag, Feiertag)
      - 1 fehlende Woche = unrealistisch (7 Tage zu?) → Datenfehler
    """
    df = df.sort_values(date_col).copy()

    full_range = pd.date_range(
        start=df[date_col].min(), end=df[date_col].max(), freq=freq
    )

    df = df.set_index(date_col).reindex(full_range).reset_index()
    df.columns = [date_col if c == "index" else c for c in df.columns]

    # UNTERSCHIEDLICHE FILL-STRATEGIE!
    if freq == "D":
        # Daily: Mit 0 füllen
        n_filled = df[target_col].isna().sum()
        df[target_col] = df[target_col].fillna(0)
        if n_filled > 0:
            print(f"  ✅ Filled {n_filled} daily gaps with 0 (likely closed)")

    elif freq == "W":
        # Weekly: NaN lassen → SARIMA interpoliert
        n_gaps = df[target_col].isna().sum()
        if n_gaps > 0:
            print(f"  ✅ {n_gaps} weekly gaps left as NaN (SARIMA will interpolate)")
        # NaN bleibt NaN!

    else:
        # Andere: Mit 0 (default)
        df[target_col] = df[target_col].fillna(0)
        print("  ✅ Filled with 0 (default)")

    if "unique_id" in df.columns:
        df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")

    return df


def load_and_prepare(df, store, item, freq="D"):
    """Lädt und bereitet eine Store-Item-Kombination vor."""

    # Determine date column
    if "date" in df.columns:
        date_col = "date"
    elif "week_start" in df.columns:
        date_col = "week_start"
    else:
        df = df.reset_index()
        if "date" in df.columns:
            date_col = "date"
        elif "week_start" in df.columns:
            date_col = "week_start"
        else:
            date_col = df.columns[0]

    # Filter
    ts = df[(df["store_nbr"] == store) & (df["item_nbr"] == item)].copy()

    # Format
    ts[date_col] = pd.to_datetime(ts[date_col])
    ts = ts.sort_values(date_col)

    target_col = "unit_sales" if "unit_sales" in ts.columns else "target_sales"

    ts = ts[[date_col, target_col]].rename(columns={date_col: "ds", target_col: "y"})

    ts["unique_id"] = f"store_{store}_item_{item}"

    print(f"📊 Loaded: {len(ts)} obs | {ts['ds'].min()} to {ts['ds'].max()}")
    print(f"   Mean: {ts['y'].mean():.2f}, Std: {ts['y'].std():.2f}")

    # Gap handling
    gap_info = detect_gaps(ts, date_col="ds", freq=freq)

    if gap_info["has_gaps"]:
        ts = fill_gaps(ts, date_col="ds", target_col="y", freq=freq)
        print(f"   After fill: {len(ts)} obs")

    return ts


def train_test_split(df, test_weeks=4):
    """Zeitbasierter Split."""
    cutoff = df["ds"].max() - pd.Timedelta(weeks=test_weeks)

    train = df[df["ds"] <= cutoff].copy()
    test = df[df["ds"] > cutoff].copy()

    print(f"✂️  Train: {len(train)} obs | Test: {len(test)} obs")

    return train, test


print("✅ Helper functions geladen")

✅ Helper functions geladen


In [4]:
# DATEN LADEN

PROJECT_ROOT = Path("..").resolve()
os.chdir(f"{PROJECT_ROOT}")

dfs = build_dataframes()

daily_smooth = dfs["smooth_daily"]  # Smooth  aus daily  Matrix
daily_erratic = dfs["erratic_daily"]  # Erratic aus daily  Matrix
weekly_smooth = dfs["smooth_weekly"]  # Smooth  aus weekly Matrix
weekly_erratic = dfs["erratic_weekly"]  # Erratic aus weekly Matrix

print("📂 Lade Daten...")


print(f"✅ Daily Smooth:   {daily_smooth.shape}")
print(f"✅ Daily Erratic:  {daily_erratic.shape}")
print(f"✅ Weekly Smooth:  {weekly_smooth.shape}")
print(f"✅ Weekly Erratic: {weekly_erratic.shape}")

Loading fact table …
Loading forecastability matrices …
Building DataFrames …
  smooth_daily         36,560 store-item pairs     39,523,827 rows
  erratic_daily        35,933 store-item pairs     36,705,401 rows
  smooth_weekly        47,196 store-item pairs     30,042,814 rows
  erratic_weekly       19,734 store-item pairs     10,182,367 rows
📂 Lade Daten...
✅ Daily Smooth:   (39523827, 12)
✅ Daily Erratic:  (36705401, 12)
✅ Weekly Smooth:  (30042814, 12)
✅ Weekly Erratic: (10182367, 12)


In [ ]:
# BASELINE FUNCTION MIT PLOTLY


def run_baseline_plotly(
    df, pattern, store, item, freq="D", season_length=7, test_weeks=4
):
    """
    Baseline-Pipeline mit Plotly Visualisierungen.

    Parameters
    ----------
    test_weeks : int
        Anzahl Wochen für Test (unterschiedlich für daily vs. weekly!)
    """

    print("\n" + "=" * 70)
    print(f"🎯 PATTERN: {pattern.upper()}")
    print(f"   Store: {store} | Item: {item}")
    print(f"   Test Period: {test_weeks} Wochen")
    print("=" * 70)

    # 1. Prepare
    ts = load_and_prepare(df, store, item, freq=freq)

    # 2. Split
    train, test = train_test_split(ts, test_weeks=test_weeks)

    # 3. Train SARIMA
    print(f"\n🤖 Training SARIMA (season={season_length})...")
    model_sarima = StatsForecast(
        models=[AutoARIMA(season_length=season_length)],
        freq=freq,
        n_jobs=1,
    )
    model_sarima.fit(train)

    # 4. Forecast SARIMA
    horizon = len(test["ds"].unique())
    print(f"   Forecasting {horizon} periods...")
    forecasts_sarima = model_sarima.predict(h=horizon)

    test_sarima = test.merge(
        forecasts_sarima.reset_index(), on=["unique_id", "ds"], how="left"
    )

    actuals = test_sarima["y"].values
    preds_sarima = test_sarima["AutoARIMA"].values
    mask = ~np.isnan(preds_sarima) & ~np.isnan(actuals)

    mae_sarima = mean_absolute_error(actuals[mask], preds_sarima[mask])
    r2_sarima = r2_score(actuals[mask], preds_sarima[mask])
    print(f"   ✅ SARIMA MAE: {mae_sarima:.2f}")
    print(f"   ✅ SARIMA R²:  {r2_sarima:.3f}")

    # 5. Train Naive
    print(f"\n🤖 Training Seasonal Naive (season={season_length})...")
    model_naive = StatsForecast(
        models=[SeasonalNaive(season_length=season_length)],
        freq=freq,
        n_jobs=1,
    )
    model_naive.fit(train)

    forecasts_naive = model_naive.predict(h=horizon)
    test_naive = test.merge(
        forecasts_naive.reset_index(), on=["unique_id", "ds"], how="left"
    )

    preds_naive = test_naive["SeasonalNaive"].values
    mae_naive = mean_absolute_error(actuals[mask], preds_naive[mask])
    r2_naive = r2_score(actuals[mask], preds_naive[mask])
    print(f"   ✅ Naive MAE: {mae_naive:.2f}")
    print(f"   ✅ Naive R²:  {r2_naive:.3f}")

    # 6. Vergleich
    improvement = (mae_naive - mae_sarima) / mae_naive * 100

    print("\n📊 RESULTS:")
    print(f"   SARIMA MAE:     {mae_sarima:.2f}")
    print(f"   Naive MAE:      {mae_naive:.2f}")
    print(f"   Improvement:    {improvement:+.1f}%")

    # 7. PLOTLY VISUALIZATIONS

    # Plot 1: Overview + Zoom
    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=(
            f"{pattern.upper()} - Forecast Comparison (Full Timeline)",
            f"Test Period Zoom (Improvement: {improvement:+.1f}%)",
        ),
        vertical_spacing=0.12,
        row_heights=[0.5, 0.5],
    )

    # Row 1: Full
    fig.add_trace(
        go.Scatter(
            x=train["ds"],
            y=train["y"],
            mode="lines",
            name="Train",
            line={"color": "blue", "width": 1},
            opacity=0.6,
            hovertemplate="Train<br>Date: %{x}<br>Sales: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=test_sarima["ds"],
            y=test_sarima["y"],
            mode="lines+markers",
            name="Test (Actual)",
            line={"color": "black", "width": 2},
            marker={"size": 6},
            hovertemplate="Actual<br>Date: %{x}<br>Sales: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=test_sarima["ds"],
            y=test_sarima["AutoARIMA"],
            mode="lines+markers",
            name=f"SARIMA (MAE={mae_sarima:.2f})",
            line={"color": "red", "width": 2, "dash": "dash"},
            marker={"size": 5, "symbol": "square"},
            hovertemplate="SARIMA<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=test_naive["ds"],
            y=test_naive["SeasonalNaive"],
            mode="lines+markers",
            name=f"Naive (MAE={mae_naive:.2f})",
            line={"color": "orange", "width": 1.5, "dash": "dot"},
            marker={"size": 4, "symbol": "diamond"},
            hovertemplate="Naive<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    # Row 2: Zoom
    fig.add_trace(
        go.Scatter(
            x=test_sarima["ds"],
            y=test_sarima["y"],
            mode="lines+markers",
            name="Actual",
            line={"color": "black", "width": 3},
            marker={"size": 8},
            showlegend=False,
            hovertemplate="Actual<br>Date: %{x}<br>Sales: %{y:.2f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=test_sarima["ds"],
            y=test_sarima["AutoARIMA"],
            mode="lines+markers",
            name="SARIMA",
            line={"color": "red", "width": 2, "dash": "dash"},
            marker={"size": 7, "symbol": "square"},
            showlegend=False,
            hovertemplate="SARIMA<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=test_naive["ds"],
            y=test_naive["SeasonalNaive"],
            mode="lines+markers",
            name="Naive",
            line={"color": "orange", "width": 1.5, "dash": "dot"},
            marker={"size": 6, "symbol": "diamond"},
            showlegend=False,
            hovertemplate="Naive<br>Date: %{x}<br>Forecast: %{y:.2f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Sales", row=1, col=1)
    fig.update_yaxes(title_text="Sales", row=2, col=1)

    fig.update_layout(
        height=800,
        template=TEMPLATE,
        hovermode="x unified",
        showlegend=True,
        legend={
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "right",
            "x": 1,
        },
    )

    fig.show()

    # Plot 2: Residuals
    residuals_sarima = test_sarima["y"] - test_sarima["AutoARIMA"]
    residuals_naive = test_naive["y"] - test_naive["SeasonalNaive"]

    fig2 = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Residuals Over Time", "Residual Distribution"),
        specs=[[{"type": "scatter"}, {"type": "histogram"}]],
    )

    fig2.add_trace(
        go.Scatter(
            x=test_sarima["ds"],
            y=residuals_sarima,
            mode="markers",
            name="SARIMA",
            marker={"size": 8, "color": "red", "opacity": 0.7},
            hovertemplate="SARIMA Residual<br>Date: %{x}<br>Error: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig2.add_trace(
        go.Scatter(
            x=test_naive["ds"],
            y=residuals_naive,
            mode="markers",
            name="Naive",
            marker={"size": 6, "color": "orange", "opacity": 0.5},
            hovertemplate="Naive Residual<br>Date: %{x}<br>Error: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    fig2.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5, row=1, col=1)

    fig2.add_trace(
        go.Histogram(
            x=residuals_sarima.dropna(),
            name="SARIMA",
            marker_color="red",
            opacity=0.7,
            nbinsx=15,
        ),
        row=1,
        col=2,
    )

    fig2.add_trace(
        go.Histogram(
            x=residuals_naive.dropna(),
            name="Naive",
            marker_color="orange",
            opacity=0.5,
            nbinsx=15,
        ),
        row=1,
        col=2,
    )

    fig2.add_vline(x=0, line_dash="dash", line_color="black", opacity=0.5, row=1, col=2)

    fig2.update_xaxes(title_text="Date", row=1, col=1)
    fig2.update_xaxes(title_text="Residual", row=1, col=2)
    fig2.update_yaxes(title_text="Residual (Actual - Predicted)", row=1, col=1)
    fig2.update_yaxes(title_text="Frequency", row=1, col=2)

    fig2.update_layout(
        height=400,
        template=TEMPLATE,
        barmode="overlay",
        showlegend=True,
        title_text=f"{pattern.upper()} - Residual Analysis",
    )

    fig2.show()

    return {
        "pattern": pattern,
        "store": store,
        "item": item,
        "test_weeks": test_weeks,
        "n_train": len(train),
        "n_test": len(test),
        "mae_sarima": mae_sarima,
        "mae_naive": mae_naive,
        "r2_sarima": r2_sarima,
        "r2_naive": r2_naive,
        "improvement_pct": improvement,
    }


print("✅ Baseline function (Plotly) geladen")

✅ Baseline function (Plotly) geladen


## Run Daily Smooth (4 Wochen Test)

In [6]:
daily_smooth_clean = daily_smooth[daily_smooth["date"] < "2016-08-22"].copy()

In [7]:
result_daily_smooth = run_baseline_plotly(
    df=daily_smooth_clean,
    pattern="daily_smooth",
    store=ITEMS_TO_MODEL["daily_smooth"]["store"],
    item=ITEMS_TO_MODEL["daily_smooth"]["item"],
    freq="D",
    season_length=7,
    test_weeks=TEST_WEEKS_DAILY,
)


🎯 PATTERN: DAILY_SMOOTH
   Store: 25 | Item: 115611
   Test Period: 4 Wochen
📊 Loaded: 1284 obs | 2013-01-01 00:00:00 to 2016-08-21 00:00:00
   Mean: 10.02, Std: 6.27
  ⚠️  Gaps: 45 dates (3.4%)
  ✅ Filled 45 daily gaps with 0 (likely closed)
   After fill: 1329 obs
✂️  Train: 1301 obs | Test: 28 obs

🤖 Training SARIMA (season=7)...


/var/folders/j7/h8mm8x6x3731n47qyx6zmky40000gn/T/ipykernel_6016/3436434363.py:74: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["unique_id"] = df["unique_id"].fillna(method="ffill").fillna(method="bfill")


   Forecasting 28 periods...
   ✅ SARIMA MAE: 3.94
   ✅ SARIMA R²:  -0.032

🤖 Training Seasonal Naive (season=7)...
   ✅ Naive MAE: 4.61
   ✅ Naive R²:  -0.597

📊 RESULTS:
   SARIMA MAE:     3.94
   Naive MAE:      4.61
   Improvement:    +14.6%


## Run Daily Erratic (4 Wochen Test)

In [8]:
result_daily_erratic = run_baseline_plotly(
    df=daily_erratic,
    pattern="daily_erratic",
    store=ITEMS_TO_MODEL["daily_erratic"]["store"],
    item=ITEMS_TO_MODEL["daily_erratic"]["item"],
    freq="D",
    season_length=7,
    test_weeks=TEST_WEEKS_DAILY,
)


🎯 PATTERN: DAILY_ERRATIC
   Store: 44 | Item: 103520
   Test Period: 4 Wochen
📊 Loaded: 1580 obs | 2013-01-02 00:00:00 to 2017-08-15 00:00:00
   Mean: 8.73, Std: 7.26
  ⚠️  Gaps: 107 dates (6.3%)
  ✅ Filled 107 daily gaps with 0 (likely closed)
   After fill: 1687 obs
✂️  Train: 1659 obs | Test: 28 obs

🤖 Training SARIMA (season=7)...


/var/folders/j7/h8mm8x6x3731n47qyx6zmky40000gn/T/ipykernel_6016/3436434363.py:74: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



   Forecasting 28 periods...
   ✅ SARIMA MAE: 2.58
   ✅ SARIMA R²:  -0.154

🤖 Training Seasonal Naive (season=7)...
   ✅ Naive MAE: 2.82
   ✅ Naive R²:  -0.401

📊 RESULTS:
   SARIMA MAE:     2.58
   Naive MAE:      2.82
   Improvement:    +8.6%


## Run Weekly Smooth (52 Wochen Test)

In [9]:
result_weekly_smooth = run_baseline_plotly(
    df=weekly_smooth,
    pattern="weekly_smooth",
    store=ITEMS_TO_MODEL["weekly_smooth"]["store"],
    item=ITEMS_TO_MODEL["weekly_smooth"]["item"],
    freq="W-MON",  # <- wichtig!
    season_length=52,
    test_weeks=TEST_WEEKS_WEEKLY,
)


🎯 PATTERN: WEEKLY_SMOOTH
   Store: 24 | Item: 1503844
   Test Period: 52 Wochen
📊 Loaded: 993 obs | 2014-01-02 00:00:00 to 2017-08-15 00:00:00
   Mean: 248.84, Std: 69.46
  ⚠️  Gaps: 45 dates (23.8%)
  ✅ Filled with 0 (default)
   After fill: 189 obs
✂️  Train: 137 obs | Test: 52 obs

🤖 Training SARIMA (season=52)...
   Forecasting 52 periods...
   ✅ SARIMA MAE: 64.18
   ✅ SARIMA R²:  -1.090

🤖 Training Seasonal Naive (season=52)...
   ✅ Naive MAE: 58.68
   ✅ Naive R²:  -0.689

📊 RESULTS:
   SARIMA MAE:     64.18
   Naive MAE:      58.68
   Improvement:    -9.4%


/var/folders/j7/h8mm8x6x3731n47qyx6zmky40000gn/T/ipykernel_6016/3436434363.py:74: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



## Run Weekly Erratic (52 Wochen Test)

In [10]:
weekly_smooth[
    (weekly_smooth["store_nbr"] == 24) & (weekly_smooth["item_nbr"] == 1503844)
].head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,year,dow,year_iso,week,week_start,month
3118118,16354239,2014-01-02,24,1503844,360.946,<NA>,2014,3,2014,1,2013-12-30,2014-01-01
3135132,16420364,2014-01-03,24,1503844,375.270,<NA>,2014,4,2014,1,2013-12-30,2014-01-01
3152416,16487079,2014-01-04,24,1503844,314.622,<NA>,2014,5,2014,1,2013-12-30,2014-01-01
3170251,16555059,2014-01-05,24,1503844,107.405,<NA>,2014,6,2014,1,2013-12-30,2014-01-01
3187489,16620829,2014-01-06,24,1503844,166.595,<NA>,2014,0,2014,2,2014-01-06,2014-01-01


In [26]:
weekly_erratic_agg = (
    weekly_erratic[
        (weekly_erratic["store_nbr"] == 51) & (weekly_erratic["item_nbr"] == 1239986)
    ]
    .groupby("week_start", as_index=False)["unit_sales"]
    .sum()
)

weekly_erratic_agg.head()

,week_start,unit_sales
0,2013-11-04,18.980
1,2013-11-11,27.349
2,2013-11-18,19.391
3,2013-11-25,32.271
4,2013-12-02,28.517


In [27]:
weekly_erratic[
    (weekly_erratic["store_nbr"] == 51) & (weekly_erratic["item_nbr"] == 1239986)
].head(20)

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,year,dow,year_iso,week,week_start,month
1054663,13598616,2013-11-06,51,1239986,18.980,<NA>,2013,2,2013,45,2013-11-04,2013-11-01
1079785,13940011,2013-11-13,51,1239986,27.349,<NA>,2013,2,2013,46,2013-11-11,2013-11-01
1105022,14282731,2013-11-20,51,1239986,19.391,<NA>,2013,2,2013,47,2013-11-18,2013-11-01
1130028,14624281,2013-11-27,51,1239986,32.271,<NA>,2013,2,2013,48,2013-11-25,2013-11-01
1157530,14980778,2013-12-04,51,1239986,13.756,<NA>,2013,2,2013,49,2013-12-02,2013-12-01
1165115,15080049,2013-12-06,51,1239986,14.761,<NA>,2013,4,2013,49,2013-12-02,2013-12-01
1184894,15335705,2013-12-11,51,1239986,1173.600,<NA>,2013,2,2013,50,2013-12-09,2013-12-01
1214170,15695852,2013-12-18,51,1239986,15.983,<NA>,2013,2,2013,51,2013-12-16,2013-12-01
1237198,15959673,2013-12-23,51,1239986,18.194,<NA>,2013,0,2013,52,2013-12-23,2013-12-01
1278829,16455312,2014-01-03,51,1239986,21.159,<NA>,2014,4,2014,1,2013-12-30,2014-01-01


In [12]:
result_weekly_erratic = run_baseline_plotly(
    df=weekly_erratic,
    pattern="weekly_erratic",
    store=ITEMS_TO_MODEL["weekly_erratic"]["store"],
    item=ITEMS_TO_MODEL["weekly_erratic"]["item"],
    freq="W-WED",
    season_length=52,
    test_weeks=TEST_WEEKS_WEEKLY,
)

/var/folders/j7/h8mm8x6x3731n47qyx6zmky40000gn/T/ipykernel_6016/3436434363.py:74: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.




🎯 PATTERN: WEEKLY_ERRATIC
   Store: 51 | Item: 1239986
   Test Period: 52 Wochen
📊 Loaded: 184 obs | 2013-11-06 00:00:00 to 2017-08-04 00:00:00
   Mean: 1189.83, Std: 1413.94
  ⚠️  Gaps: 118 dates (60.2%)
  ✅ Filled with 0 (default)
   After fill: 196 obs
✂️  Train: 144 obs | Test: 52 obs

🤖 Training SARIMA (season=52)...
   Forecasting 52 periods...
   ✅ SARIMA MAE: 68.84
   ✅ SARIMA R²:  0.000

🤖 Training Seasonal Naive (season=52)...
   ✅ Naive MAE: 50.56
   ✅ Naive R²:  0.000

📊 RESULTS:
   SARIMA MAE:     68.84
   Naive MAE:      50.56
   Improvement:    -36.2%


## Summary Dashboard

In [ ]:
# Sammle Ergebnisse
results = [
    result_daily_smooth,
    result_daily_erratic,
    result_weekly_smooth,
    result_weekly_erratic,
]

summary_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("📊 SUMMARY - ALL PATTERNS")
print("=" * 70)
print(summary_df.to_string(index=False))

# Zeige Test Weeks
print("\n" + "=" * 70)
print("TEST PERIODS VERWENDET:")
print("=" * 70)
for _, row in summary_df.iterrows():
    print(
        f"  {row['pattern']:20} → {row['test_weeks']} Wochen ({row['n_test']} Perioden)"
    )

# Interactive Bar Chart
fig = go.Figure()

x = summary_df["pattern"]

fig.add_trace(
    go.Bar(
        x=x,
        y=summary_df["mae_sarima"],
        name="SARIMA",
        marker_color="red",
        opacity=0.8,
        hovertemplate="%{x}<br>SARIMA MAE: %{y:.2f}<extra></extra>",
        text=summary_df["mae_sarima"].round(2),
        textposition="outside",
    )
)

fig.add_trace(
    go.Bar(
        x=x,
        y=summary_df["mae_naive"],
        name="Naive",
        marker_color="orange",
        opacity=0.6,
        hovertemplate="%{x}<br>Naive MAE: %{y:.2f}<extra></extra>",
        text=summary_df["mae_naive"].round(2),
        textposition="outside",
    )
)

# Improvement annotations
for i, row in summary_df.iterrows():
    fig.add_annotation(
        x=i,
        y=max(row["mae_sarima"], row["mae_naive"]) + 2,
        text=f"{row['improvement_pct']:+.1f}%",
        showarrow=False,
        font={
            "size": 14,
            "color": "green" if row["improvement_pct"] > 0 else "red",
            "family": "Arial Black",
        },
    )

fig.update_layout(
    title="Baseline Model Comparison - MAE by Pattern<br><sub>Daily: 4 Wochen Test | Weekly: 52 Wochen Test</sub>",
    xaxis_title="Pattern",
    yaxis_title="Mean Absolute Error (MAE)",
    barmode="group",
    template=TEMPLATE,
    height=500,
    hovermode="x unified",
    showlegend=True,
)

fig.show()

# Save
summary_df.to_csv("baseline_summary.csv", index=False)
print("\n✅ Summary saved to: baseline_summary.csv")


📊 SUMMARY - ALL PATTERNS
       pattern  store    item  test_weeks  n_train  n_test  mae_sarima  mae_naive  r2_sarima  r2_naive  improvement_pct
  daily_smooth     25  115611           4     1301      28    3.935840   4.607143  -0.032048 -0.596533        14.570920
 daily_erratic     44  103520           4     1659      28    2.578132   2.821429  -0.153654 -0.401477         8.623171
 weekly_smooth     24 1503844          52      137      52   64.176867  58.677365  -1.090212 -0.688508        -9.372442
weekly_erratic     51 1239986          52      144      52   68.837301  50.558846   0.000000  0.000000       -36.152832

TEST PERIODS VERWENDET:
  daily_smooth         → 4 Wochen (28 Perioden)
  daily_erratic        → 4 Wochen (28 Perioden)
  weekly_smooth        → 52 Wochen (52 Perioden)
  weekly_erratic       → 52 Wochen (52 Perioden)



✅ Summary saved to: baseline_summary.csv
